In [3]:
# Reliance option chain from NSE. Yahoo Finance lists no Indian option contracts.
# curl_cffi ships with yfinance; a plain requests call is rejected by NSE.

from curl_cffi import requests
import pandas as pd
import yfinance

SYMBOL = "RELIANCE"

session = requests.Session(impersonate="chrome")
session.get("https://www.nseindia.com/option-chain", timeout=30)
headers = {
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.nseindia.com/option-chain",
}

expiries = session.get(
    "https://www.nseindia.com/api/option-chain-contract-info",
    params={"symbol": SYMBOL},
    headers=headers,
    timeout=30,
).json()["expiryDates"]
expiry = expiries[0]

payload = session.get(
    "https://www.nseindia.com/api/option-chain-v3",
    params={"type": "Equity", "symbol": SYMBOL, "expiry": expiry},
    headers=headers,
    timeout=30,
).json()["records"]

rows = []
for item in payload["data"]:
    for side in ("CE", "PE"):
        leg = item.get(side)
        if not leg:
            continue
        rows.append(
            {
                "expiry": expiry,
                "strike": item["strikePrice"],
                "side": side,
                "last": leg["lastPrice"],
                "bid": leg["buyPrice1"],
                "ask": leg["sellPrice1"],
                "iv": leg["impliedVolatility"],
                "oi": leg["openInterest"],
                "volume": leg["totalTradedVolume"],
            }
        )

options = pd.DataFrame(rows)
spot = float(payload["underlyingValue"])
yahoo = float(yfinance.Ticker(f"{SYMBOL}.NS").fast_info.last_price)

near = options[options["strike"].between(spot * 0.97, spot * 1.03)]
calls = near[near["side"] == "CE"].sort_values("strike")
puts = near[near["side"] == "PE"].sort_values("strike")

quoted = options[(options["bid"] > 0) & (options["ask"] >= options["bid"])]
book = quoted.pivot(index="strike", columns="side")
call_mid = (book["bid"]["CE"] + book["ask"]["CE"]) / 2
put_mid = (book["bid"]["PE"] + book["ask"]["PE"]) / 2
synthetic = book.index.to_series() + call_mid - put_mid
band = synthetic[synthetic.index.to_series().between(spot * 0.95, spot * 1.05)].dropna()

spot_gap = abs(spot - yahoo) / yahoo
calls_fall = bool(calls["last"].is_monotonic_decreasing)
puts_rise = bool(puts["last"].is_monotonic_increasing)
synthetic_span = float(band.max() - band.min())
checks = {
    "rows": len(options) > 0,
    "spot_matches_yahoo": spot_gap < 0.01,
    "calls_fall_with_strike": calls_fall,
    "puts_rise_with_strike": puts_rise,
    "put_call_synthetic_flat": synthetic_span < spot * 0.01,
}

print(f"{SYMBOL} {expiry} as of {payload['timestamp']}")
print(f"NSE spot {spot:.2f}  Yahoo {yahoo:.2f}  gap {spot_gap:.3%}")
print(f"strikes {options['strike'].nunique()}  rows {len(options)}  expiries {expiries}")
print(f"synthetic forward {float(band.median()):.2f}  span {synthetic_span:.2f}")
print(checks)
if not all(checks.values()):
    raise AssertionError(checks)

options.sort_values(["strike", "side"]).reset_index(drop=True)


RELIANCE 27-Oct-2026 as of 06-Oct-2026 10:55:33
NSE spot 1210.20  Yahoo 1210.20  gap 0.000%
strikes 48  rows 96  expiries ['27-Oct-2026', '23-Nov-2026', '29-Dec-2026']
synthetic forward 1213.47  span 1.28
{'rows': True, 'spot_matches_yahoo': True, 'calls_fall_with_strike': True, 'puts_rise_with_strike': True, 'put_call_synthetic_flat': True}


,expiry,strike,side,last,bid,ask,iv,oi,volume
0,27-Oct-2026,980,CE,0.00,224.00,241.45,0.00,0,0
1,27-Oct-2026,980,PE,0.00,0.15,2.95,0.00,0,0
2,27-Oct-2026,1000,CE,0.00,205.35,217.65,0.00,100,0
3,27-Oct-2026,1000,PE,0.40,0.40,0.45,36.47,583,78
4,27-Oct-2026,1020,CE,0.00,185.10,201.85,0.00,1,0
...,...,...,...,...,...,...,...,...,...
91,27-Oct-2026,1480,PE,0.00,260.55,271.70,0.00,87,0
92,27-Oct-2026,1500,CE,0.30,0.25,0.35,35.71,1071,176
93,27-Oct-2026,1500,PE,286.75,284.45,285.75,52.45,2543,41
94,27-Oct-2026,1520,CE,0.20,0.15,0.20,35.59,1094,22


In [4]:
# Yearly move: sample std of daily log returns, scaled by sqrt(252).
# Yahoo's latest bar is the live price until 15:30 IST, so it is not a full day.

import numpy as np
import yfinance
from datetime import datetime
from zoneinfo import ZoneInfo

closes = yfinance.Ticker("RELIANCE.NS").history(period="1y", auto_adjust=True)["Close"].dropna()
now = datetime.now(ZoneInfo("Asia/Kolkata"))
session_close = now.replace(hour=15, minute=30, second=0, microsecond=0)
if closes.index[-1].date() == now.date() and now < session_close:
    closes = closes.iloc[:-1]

returns = np.log(closes).diff().dropna()

def yearly_move(days):
    window = returns.iloc[-days:]
    if len(window) < days:
        raise LookupError(f"need {days} completed sessions, have {len(window)}")
    return float(window.std(ddof=1) * np.sqrt(252))

move_20 = yearly_move(20)
move_60 = yearly_move(60)
hedge_forecast = 0.70 * move_20 + 0.30 * move_60

print(f"last completed close {closes.index[-1].date()}  {closes.iloc[-1]:.2f}")
print(f"20-day yearly move {move_20:.2%}")
print(f"60-day yearly move {move_60:.2%}")
print(f"hedge forecast  {hedge_forecast:.2%}")


last completed close 2026-10-05  1186.40
20-day yearly move 18.97%
60-day yearly move 18.76%
hedge forecast  18.91%


In [5]:
# hedge_forecast is a fraction. NSE iv and the 3-point band are in percent.

strikes = options["strike"].drop_duplicates()
atm_strike = float(strikes.iloc[(strikes - spot).abs().argmin()])
atm = options[options["strike"].eq(atm_strike)].set_index("side")
quoted_iv = atm.loc[atm["iv"] > 0, "iv"]
if quoted_iv.empty:
    raise LookupError(f"no implied vol at {atm_strike}")
atm_iv = float(quoted_iv.mean())

forecast_rv = hedge_forecast * 100
edge = forecast_rv - atm_iv
if edge < 0:
    trade = "short straddle"
elif edge > 3:
    trade = "long straddle"
else:
    trade = "no trade"

print(f"ATM {atm_strike:.0f}  call iv {atm.loc['CE', 'iv']:.2f}  put iv {atm.loc['PE', 'iv']:.2f}")
print(f"forecast rv {forecast_rv:.2f}  atm iv {atm_iv:.2f}  edge {edge:+.2f}")
print(trade)
atm


ATM 1210  call iv 21.67  put iv 24.88
forecast rv 18.91  atm iv 23.27  edge -4.37
short straddle


,expiry,strike,last,bid,ask,iv,oi,volume
side,,,,,,,,
CE,27-Oct-2026,1210,29.25,29.05,29.15,21.67,3975,4030
PE,27-Oct-2026,1210,25.20,25.30,25.45,24.88,3876,1695


In [6]:
from backtest.vol_arb_backtest import run

summary, trades = run()
print(summary)
trades


{'sessions': 555, 'first': '2024-07-08', 'last': '2026-10-05', 'trades': 77, 'shorts': 56, 'longs': 21, 'net_points': -22.15894300000047, 'gross_points': 14.449999999999541, 'cost_points': 36.60894300000001, 'net_rupees': -19016.421125000226, 'average_net_points': -0.2877784805194866, 'median_hold_sessions': 2.0, 'one_session_holds': 28, 'hedge_trades': 41, 'exits': {'iv': 44, 'gap': 19, 'expiry': 14}, 'open_points': 3.3648499999999935}


,entry,exit,direction,expiry,strike,premium,entry_forecast,entry_edge,reason,hold_sessions,gross_points,cost_points,net_points,net_rupees,hedge_trades,futures_lots_at_exit,spot_entry,spot_exit
0,2024-10-08,2024-10-09,short,2024-10-31,2740.0,137.20,21.960886,-0.532459,iv,1,9.25,0.518638,8.731362,2182.840375,0,0,2794.70,2749.25
1,2024-10-16,2024-10-25,long,2024-10-31,2680.0,86.60,22.962795,5.960013,iv,7,24.25,1.233563,23.016437,5754.109250,2,0,2708.15,2655.70
2,2024-10-29,2024-10-31,long,2024-10-31,1330.0,17.00,21.896187,3.286700,expiry,2,-3.80,0.581617,-4.381617,-2190.808500,1,-1,1340.00,1332.10
3,2024-11-01,2024-11-04,short,2024-11-28,1330.0,69.95,16.917112,-7.188082,gap,1,2.25,0.260781,1.989219,994.609500,0,0,1338.70,1302.15
4,2024-11-05,2024-11-11,short,2024-11-28,1300.0,63.05,18.434501,-6.207715,iv,4,15.50,0.253276,15.246723,7623.361750,0,0,1305.30,1272.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,2026-07-23,2026-07-24,short,2026-07-28,1290.0,26.40,19.788146,-0.707672,iv,1,6.45,0.228998,6.221002,3110.500750,0,0,1272.70,1278.00
73,2026-07-29,2026-07-30,short,2026-08-25,1270.0,59.45,19.343715,-2.222152,iv,1,1.55,0.279712,1.270288,635.144000,0,0,1275.90,1293.10
74,2026-08-06,2026-08-07,short,2026-08-25,1280.0,66.00,18.937245,-1.506983,iv,1,3.35,1.078901,2.271099,1135.549750,1,1,1325.00,1334.72
75,2026-08-19,2026-08-24,long,2026-08-25,1320.0,21.75,19.131852,3.073371,iv,3,-3.55,0.216753,-3.766752,-1883.376250,0,0,1311.00,1309.74
